# Data Ingestion — Card & Krueger (1994)

Reads `public.dat` (fixed-width ASCII, 410 stores) and writes a clean CSV to `data/processed/`.
Column names, positions, and encoding decisions are documented here and nowhere else.
All other notebooks read from the processed CSV.

In [59]:
import pandas as pd

## Column definitions

Column positions are 0-based, end-exclusive, taken directly from the codebook.
Names follow these conventions:
- `is_` prefix for binary (0/1) flags
- `has_` prefix for binary feature indicators
- `n_` prefix for counts
- `_wave2` suffix for post-policy (Wave 2) versions of Wave 1 variables

In [60]:
col_names = [
    # Store identifiers & characteristics
    'store_id', 'chain_id', 'is_company_owned', 'is_nj',
    # Location dummies
    'is_south_nj', 'is_central_nj', 'is_north_nj', 'is_pa_philly_suburbs', 'is_pa_easton', 'is_shore',
    # Wave 1 (pre-policy, Feb/Mar 1992) variables
    'n_emp_fulltime', 'n_emp_parttime', 'n_managers', 'wage_start', 'months_to_raise',
    'first_raise', 'has_bonus', 'pct_affected', 'meals', 'open_hour', 'hrs_open',
    'price_soda', 'price_fry', 'price_entree', 'n_registers', 'n_registers_11am',
    # Wave 2 (post-policy, Nov/Dec 1992) variables
    'interview_type_wave2', 'status_wave2', 'date_wave2',
    'n_emp_fulltime_wave2', 'n_emp_parttime_wave2', 'n_managers_wave2', 'wage_start_wave2', 'months_to_raise_wave2',
    'first_raise_wave2', 'has_special_program_wave2', 'meals_wave2', 'open_hour_wave2', 'hrs_open_wave2',
    'price_soda_wave2', 'price_fry_wave2', 'price_entree_wave2', 'n_registers_wave2', 'n_registers_11am_wave2',
]

colspecs = [
    (0, 3), (4, 5), (6, 7), (8, 9),
    (10, 11), (12, 13), (14, 15), (16, 17), (18, 19), (20, 21),
    (25, 30), (31, 36), (37, 42), (43, 48), (49, 54),
    (55, 60), (61, 62), (63, 68), (69, 70), (71, 76), (77, 82),
    (83, 88), (89, 94), (95, 100), (101, 103), (104, 106),
    (107, 108), (109, 110), (111, 117),
    (121, 126), (127, 132), (133, 138), (139, 144), (145, 150),
    (151, 156), (157, 158), (159, 160), (161, 166), (167, 172),
    (173, 178), (179, 184), (185, 190), (191, 193), (194, 196),
]

df = pd.read_fwf(
    '../data/raw/public.dat',
    colspecs=colspecs,
    names=col_names,
    na_values='.',
    header=None,
)

print(f"Loaded {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

Loaded 410 rows × 44 columns


,store_id,chain_id,is_company_owned,is_nj,is_south_nj,is_central_nj,is_north_nj,is_pa_philly_suburbs,is_pa_easton,is_shore,...,first_raise_wave2,has_special_program_wave2,meals_wave2,open_hour_wave2,hrs_open_wave2,price_soda_wave2,price_fry_wave2,price_entree_wave2,n_registers_wave2,n_registers_11am_wave2
0,46,1,0,0,0,0,0,1,0,0,...,0.08,1.0,2.0,6.5,16.5,1.03,NaN,0.94,4.0,4.0
1,49,2,0,0,0,0,0,1,0,0,...,0.05,0.0,2.0,10.0,13.0,1.01,0.89,2.35,4.0,4.0
2,506,2,1,0,0,0,0,1,0,0,...,0.25,NaN,1.0,11.0,11.0,0.95,0.74,2.33,4.0,3.0
3,56,4,1,0,0,0,0,1,0,0,...,0.15,0.0,2.0,10.0,12.0,0.92,0.79,0.87,2.0,2.0
4,61,4,1,0,0,0,0,1,0,0,...,0.15,0.0,2.0,10.0,12.0,1.01,0.84,0.95,2.0,2.0


In [61]:
region_cols = [
    'is_south_nj', 'is_central_nj', 'is_north_nj',
    'is_pa_philly_suburbs', 'is_pa_easton', 'is_shore',
]
region_labels = ['south_nj', 'central_nj', 'north_nj', 'pa_philly_suburbs', 'pa_easton', 'shore']

df['region'] = (
    df[region_cols]
    .idxmax(axis=1)
    .map(dict(zip(region_cols, region_labels)))
)
df = df.drop(columns=region_cols)

chain_map = {1: 'Burger King', 2: 'KFC', 3: 'Roy Rogers', 4: "Wendy's"}
df['chain'] = df['chain_id'].map(chain_map)
df = df.drop(columns=['chain_id'])

print(df['region'].value_counts())
print()
print(df['chain'].value_counts())

region
north_nj             175
south_nj              93
central_nj            63
pa_easton             43
pa_philly_suburbs     36
Name: count, dtype: int64

chain
Burger King    171
Roy Rogers      99
KFC             80
Wendy's         60
Name: count, dtype: int64


## Filter to Active Stores

Keep only stores that completed the Wave 2 interview (`status_wave2 == 1`).
The remaining codes are: 0=refused, 2=renovation, 3=permanently closed, 4=highway construction, 5=mall fire.
After filtering, `status_wave2` is constant and uninformative, so it is dropped.

In [62]:
n_before = len(df)
df = df[df['status_wave2'] == 1].drop(columns=['status_wave2']).reset_index(drop=True)
n_dropped = n_before - len(df)

print(f"Dropped {n_dropped} stores that did not complete Wave 2 interview")
print(f"Remaining: {len(df)} stores")

Dropped 11 stores that did not complete Wave 2 interview
Remaining: 399 stores


## Export to CSV

In [63]:
output_path = '../data/processed/card_krueger_1994.csv'

df.to_csv(output_path, index=False)

print(f"Saved {len(df)} rows to {output_path}")

Saved 399 rows to ../data/processed/card_krueger_1994.csv


## Reshape to Long Format

For DiD regression we need one row per **store × wave** with a single `is_post` indicator.
Columns that only exist in one wave are included with NaN in the other:
- Wave 1 only: `has_bonus`, `pct_affected`
- Wave 2 only: `interview_type`, `date`, `has_special_program`

In [64]:
# Store-level invariants — same value in both wave rows
id_cols = [
    'store_id', 'chain', 'is_company_owned', 'is_nj', 'region',
]

# Variables that exist in both waves under matching names
paired_cols = [
    'n_emp_fulltime', 'n_emp_parttime', 'n_managers',
    'wage_start', 'months_to_raise', 'first_raise', 'meals',
    'open_hour', 'hrs_open', 'price_soda', 'price_fry', 'price_entree',
    'n_registers', 'n_registers_11am',
]

# Wave 1 rows (pre-policy)
wave1 = df[id_cols + paired_cols].copy()
wave1['is_post']      = 0
wave1['has_bonus']    = df['has_bonus']
wave1['pct_affected'] = df['pct_affected']

# Wave 2 rows (post-policy) — rename _wave2 columns to base names
wave2 = df[id_cols].copy()
for col in paired_cols:
    wave2[col] = df[col + '_wave2']
wave2['is_post']             = 1
wave2['interview_type']      = df['interview_type_wave2']
wave2['date']                = df['date_wave2']
wave2['has_special_program'] = df['has_special_program_wave2']

# Stack and sort by store then wave
df_long = pd.concat([wave1, wave2], ignore_index=True)
df_long = df_long.sort_values(['store_id', 'is_post']).reset_index(drop=True)

# Put the DiD key columns together
remaining = [c for c in df_long.columns if c not in ('store_id', 'is_nj', 'is_post')]
df_long = df_long[['store_id', 'is_nj', 'is_post'] + remaining]

print(f"Long format: {df_long.shape[0]} rows × {df_long.shape[1]} columns")
df_long.head(6)

Long format: 798 rows × 25 columns


,store_id,is_nj,is_post,chain,is_company_owned,region,n_emp_fulltime,n_emp_parttime,n_managers,wage_start,...,price_soda,price_fry,price_entree,n_registers,n_registers_11am,has_bonus,pct_affected,interview_type,date,has_special_program
0,1,1,0,Burger King,0,central_nj,16.0,30.0,4.0,4.50,...,0.93,0.83,0.85,4.0,3.0,0.0,80.0,NaN,NaN,NaN
1,1,1,1,Burger King,0,central_nj,20.0,40.0,4.0,5.05,...,1.05,0.79,0.90,4.0,3.0,NaN,NaN,1.0,110592.0,0.0
2,2,1,0,Burger King,0,central_nj,10.0,6.0,3.0,4.75,...,1.06,0.91,0.96,2.0,2.0,1.0,80.0,NaN,NaN,NaN
3,2,1,1,Burger King,0,central_nj,7.5,10.0,3.0,5.25,...,1.05,1.01,0.94,2.0,2.0,NaN,NaN,1.0,110592.0,1.0
4,3,1,0,KFC,0,central_nj,6.0,13.0,3.0,4.25,...,1.06,0.95,3.09,5.0,3.0,1.0,50.0,NaN,NaN,NaN
5,3,1,1,KFC,0,central_nj,4.0,7.0,3.0,5.05,...,1.05,0.94,2.75,5.0,3.0,NaN,NaN,1.0,111792.0,0.0


In [65]:
# Card & Krueger's FTE measure: part-timers count as 0.5
df_long['n_fte'] = df_long['n_emp_fulltime'] + 0.5 * df_long['n_emp_parttime'] + df_long['n_managers']

In [66]:
long_output_path = '../data/processed/card_krueger_1994_long.csv'

df_long.to_csv(long_output_path, index=False)

print(f"Saved {len(df_long)} rows to {long_output_path}")

Saved 798 rows to ../data/processed/card_krueger_1994_long.csv


In [67]:
df_long.columns

Index(['store_id', 'is_nj', 'is_post', 'chain', 'is_company_owned', 'region',
       'n_emp_fulltime', 'n_emp_parttime', 'n_managers', 'wage_start',
       'months_to_raise', 'first_raise', 'meals', 'open_hour', 'hrs_open',
       'price_soda', 'price_fry', 'price_entree', 'n_registers',
       'n_registers_11am', 'has_bonus', 'pct_affected', 'interview_type',
       'date', 'has_special_program', 'n_fte'],
      dtype='str')